# Mini-Projeto M1: Análise Exploratória da Base Varejo

**Aluna:** Maria Pilar Aponte | **Turma:** T8

## Objetivo
Ler, diagnosticar, limpar e analisar a base Varejo (Kaggle), gerando uma versão tratada
(`varejo_limpo.csv`) e conclusões numéricas sobre o tamanho das compras e as categorias
de produtos compradas.

## Ambiente
Executado no VS Code com Python 3.14 e pandas 3.1, em ambiente virtual (`.venv`).
Roteiro baseado no exemplo da Aula 23 (Mini-Projeto Simulado), adaptado para esta base.

## Etapas
1. Leitura e diagnóstico
2. Limpeza e padronização
3. Funções e organização
4. Análise (estatísticas, filtro e agrupamentos)
5. Exportação e conclusões

# SPRINT 1: Leitura e Diagnóstico

Lemos a base e registramos sua estrutura e seus problemas de qualidade, **sem alterar nenhum dado**.
Quando uma verificação retorna zero, isso também é registrado como resultado.

O arquivo usa ponto e vírgula (`;`) como separador. Sem informar `sep=";"`, o pandas lê cada
linha inteira como uma única coluna (830.000 linhas e 1 coluna).

In [1]:
import pandas as pd

# Caminho relativo: o CSV está na mesma pasta do notebook
ARQUIVO_ENTRADA = "base_varejo.csv"
ARQUIVO_SAIDA = "varejo_limpo.csv"

# sep=";" porque o arquivo usa ponto e vírgula como separador
df_original = pd.read_csv(ARQUIVO_ENTRADA, sep=";")

display(df_original.head())

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


## Estrutura: dimensões, colunas e tipos

`shape` mostra linhas e colunas, `columns` lista os nomes e `info()` mostra o tipo de cada
coluna e quantos valores preenchidos ela tem. No pandas 3, texto aparece como `str`.

In [2]:
print("Dimensões (linhas, colunas):", df_original.shape)
print("Colunas:", df_original.columns.tolist())
print()
df_original.info()

Dimensões (linhas, colunas): (830000, 14)
Colunas: ['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
d

## O que cada linha representa

Comparamos o total de linhas com a quantidade de compras, clientes e produtos distintos.

In [3]:
print("Total de linhas:", len(df_original))
print("Compras distintas (CO_ID):", df_original["CO_ID"].nunique())
print("Clientes distintos (CL_ID):", df_original["CL_ID"].nunique())
print("Produtos distintos (PR_ID):", df_original["PR_ID"].nunique())

Total de linhas: 830000
Compras distintas (CO_ID): 18471
Clientes distintos (CL_ID): 1000
Produtos distintos (PR_ID): 229


Há muito menos compras do que linhas: **cada linha representa um item de uma compra**.
Uma mesma compra (`CO_ID`) aparece em várias linhas, uma por produto, e um mesmo cliente
(`CL_ID`) pode ter várias compras.

| Coluna | Significado |
|---|---|
| `DATA` | Data da compra (lida como texto, formato dd/mm/aaaa) |
| `CO_ID` | Identificador da compra |
| `CL_ID` | Identificador do cliente |
| `CL_GENERO` | Gênero do cliente (F / M) |
| `CL_EC` | Estado civil do cliente (código de 1 a 5) |
| `CL_FHL` | Número de filhos do cliente |
| `CL_SEG` | Segmento do cliente (A / B / C) |
| `PR_ID` | Identificador do produto |
| `PR_CAT` | Categoria do produto |
| `PR_NOME` | Nome do produto |

Os identificadores (`CO_ID`, `CL_ID`, `PR_ID`) servem só para identificar registros e
**não** são usados como medida numérica.

## Colunas sem nome e valores ausentes

O cabeçalho do arquivo termina com `;;;;`, o que gerou colunas extras chamadas `Unnamed`.
`isna().sum()` conta, por coluna, quantos valores estão vazios.

In [4]:
display(df_original.isna().sum())

DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

As 4 colunas `Unnamed` estão 100% vazias (830.000 ausentes cada).
Nas 10 colunas reais, **não há nenhum valor ausente**.

## Textos vazios e inconsistências de escrita

Um texto com apenas espaços (`" "`) não é contado pelo `isna()`. Para cada coluna de texto,
verificamos:
- textos vazios ou só com espaços;
- valores com espaço sobrando no início ou no fim;
- variações de escrita: se o número de valores distintos diminui ao padronizar
  (sem espaços e em maiúsculas), o mesmo rótulo estava escrito de formas diferentes.

In [5]:
colunas_texto = ["CL_GENERO", "CL_SEG", "PR_CAT", "PR_NOME"]

for coluna in colunas_texto:
    texto = df_original[coluna].astype("string")
    vazios = texto.str.strip().eq("").sum()
    espacos_extras = (texto != texto.str.strip()).sum()
    variacoes = texto.nunique() - texto.str.strip().str.upper().nunique()
    print(f"{coluna}: {vazios} vazios | {espacos_extras} com espaços extras | {variacoes} variações de escrita")

CL_GENERO: 0 vazios | 0 com espaços extras | 0 variações de escrita


CL_SEG: 0 vazios | 0 com espaços extras | 0 variações de escrita


PR_CAT: 0 vazios | 0 com espaços extras | 0 variações de escrita


PR_NOME: 0 vazios | 0 com espaços extras | 0 variações de escrita


In [6]:
for coluna in ["CL_GENERO", "CL_SEG", "PR_CAT"]:
    print(coluna, "→", df_original[coluna].unique())

CL_GENERO → <StringArray>
['M', 'F']
Length: 2, dtype: str
CL_SEG → <StringArray>
['C', 'B', 'A']
Length: 3, dtype: str
PR_CAT → <StringArray>
['BEBIDAS', 'HIGIENE', 'ALIMENTOS', 'LIMPEZA', 'ACESSORIOS', 'PET', '#N/D']
Length: 7, dtype: str


Não há textos vazios, espaços extras nem variações de escrita: os rótulos já estão
padronizados em maiúsculas.

Porém, `PR_CAT` contém o valor **`#N/D`**, que é um erro de fórmula do Excel (PROCV sem
correspondência). Para o pandas, `#N/D` é um texto comum, então o `isna()` não o detecta.

In [7]:
linhas_nd = df_original[df_original["PR_CAT"] == "#N/D"]

print("Linhas com #N/D em PR_CAT:", len(linhas_nd))
print("Linhas com #N/D em PR_NOME:", (df_original["PR_NOME"] == "#N/D").sum())
print("Produtos afetados (PR_ID):", linhas_nd["PR_ID"].unique())

Linhas com #N/D em PR_CAT: 3650
Linhas com #N/D em PR_NOME: 3650
Produtos afetados (PR_ID): [107]


O `#N/D` aparece em 3.650 linhas, sempre no mesmo produto (`PR_ID` 107), tanto na
categoria quanto no nome. Ou seja, é um produto cujo cadastro não foi encontrado na exportação.

## Duplicatas

`duplicated()` marca as linhas idênticas a uma linha anterior em **todas** as colunas.

In [8]:
duplicadas = df_original[df_original.duplicated()]

print("Duplicatas completas:", len(duplicadas))
print(f"Percentual da base: {len(duplicadas) / len(df_original):.1%}")
print("Compras com itens repetidos:", duplicadas["CO_ID"].nunique(),
      "de", df_original["CO_ID"].nunique())

# Exemplo: todas as ocorrências de linhas repetidas, ordenadas por compra e produto
display(
    df_original[df_original.duplicated(keep=False)]
    .sort_values(["CO_ID", "PR_ID"])
    .head(6)
)

Duplicatas completas: 96553
Percentual da base: 11.6%
Compras com itens repetidos: 14566 de 18471


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
40,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
46,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
14,01/02/2019,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,NaN,NaN,NaN,NaN
19,01/02/2019,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,NaN,NaN,NaN,NaN


As repetições estão espalhadas por 78,9% das compras e sempre envolvem a mesma compra e o
mesmo produto. Como a base não tem coluna de quantidade, esse padrão indica que o cliente
levou **mais de uma unidade** do item, e não um erro de registro.

### Resumo do diagnóstico

| Verificação | Resultado |
|---|---|
| Dimensões | 830.000 linhas e 14 colunas (10 reais + 4 `Unnamed`) |
| O que cada linha representa | Um item de compra |
| Compras / clientes / produtos distintos | 18.471 / 1.000 / 229 (com 118 nomes de produto) |
| Colunas `Unnamed` | 4, todas 100% vazias |
| Valores ausentes nas colunas reais | 0 |
| Textos vazios ou só com espaços | 0 |
| Espaços extras e variações de escrita | 0 |
| `#N/D` (erro do Excel) | 3.650 linhas, produto 107, em `PR_CAT` e `PR_NOME` |
| Duplicatas completas | 96.553 (11,6%), em 14.566 das 18.471 compras |

# SPRINT 2: Limpeza e Padronização

| Problema | Decisão | Justificativa |
|---|---|---|
| 4 colunas `Unnamed` vazias | Remover | Criadas pelos `;` extras do cabeçalho; não contêm dados |
| Textos | Padronizar (sem espaços nas pontas, maiúsculas) | Já estavam corretos; a padronização garante que futuras cargas não criem grupos duplicados |
| Textos vazios | Converter em ausente | Nenhum encontrado; a regra fica como proteção |
| `#N/D` em categoria e nome (3.650 linhas) | Substituir por `NÃO IDENTIFICADO` | O item foi comprado, então a linha é mantida; só não sabemos qual é o produto |
| 96.553 duplicatas | **Manter** | Sem coluna de quantidade, a repetição representa mais de uma unidade do mesmo item. Removê-las apagaria compras reais |

Os dados originais são preservados em `df_original`; a limpeza gera uma cópia, `df_limpo`.

# SPRINT 3: Funções e Organização

O projeto usa duas funções com parâmetro e retorno:

- **`tratar_dados(tabela, colunas_texto, remover_duplicatas)`**: recebe a base original e
  retorna uma cópia tratada, aplicando as decisões da tabela acima;
- **`gerar_resumo(tabela, coluna)`**: recebe uma tabela e o nome de uma coluna numérica e
  retorna contagem de valores válidos, média, mínimo e máximo.

Fluxo do projeto: **leitura (`df_original`) → limpeza (`tratar_dados`) → análise (`gerar_resumo` e agrupamentos)**.

In [9]:
def tratar_dados(tabela, colunas_texto, remover_duplicatas=False):
    """Recebe a base original e retorna uma cópia tratada."""
    tratado = tabela.copy()   # nunca altera a base original

    # Remove apenas colunas 100% vazias (as "Unnamed" do cabeçalho)
    tratado = tratado.dropna(axis=1, how="all")

    # Padroniza textos: tira espaços das pontas, coloca em maiúsculas
    # e transforma texto vazio em ausente
    for coluna in colunas_texto:
        tratado[coluna] = (
            tratado[coluna]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
        )

    # "#N/D" é erro de fórmula do Excel: categoria e nome desconhecidos
    for coluna in ["PR_CAT", "PR_NOME"]:
        tratado[coluna] = tratado[coluna].replace("#N/D", "NÃO IDENTIFICADO")

    # Duplicatas só são removidas se a decisão for confirmada
    if remover_duplicatas:
        tratado = tratado.drop_duplicates()

    return tratado


def gerar_resumo(tabela, coluna):
    """Retorna as quatro estatísticas obrigatórias de uma coluna numérica."""
    valores = tabela[coluna]
    return {
        "contagem_valida": int(valores.count()),   # ignora ausentes
        "media": round(float(valores.mean()), 2),
        "minimo": int(valores.min()),
        "maximo": int(valores.max()),
    }

In [10]:
# Duplicatas mantidas: representam mais de uma unidade do mesmo item
df_limpo = tratar_dados(df_original, colunas_texto, remover_duplicatas=False)

## Comparação antes e depois

In [11]:
print("Linhas:  antes", len(df_original), "| depois", len(df_limpo))
print("Colunas: antes", df_original.shape[1], "| depois", df_limpo.shape[1])
print("Duplicatas: antes", df_original.duplicated().sum(), "| depois", df_limpo.duplicated().sum())
print("#N/D em PR_CAT: antes", (df_original["PR_CAT"] == "#N/D").sum(),
      "| depois", (df_limpo["PR_CAT"] == "#N/D").sum())

print("\nAusentes depois da limpeza:")
display(df_limpo.isna().sum())

print("Categorias depois da limpeza:", df_limpo["PR_CAT"].unique())

Linhas:  antes 830000 | depois 830000
Colunas: antes 14 | depois 10


Duplicatas: antes 96553 | depois 96553
#N/D em PR_CAT: antes 3650 | depois 0

Ausentes depois da limpeza:


DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

Categorias depois da limpeza: <StringArray>
[         'BEBIDAS',          'HIGIENE',        'ALIMENTOS',
          'LIMPEZA',       'ACESSORIOS',              'PET',
 'NÃO IDENTIFICADO']
Length: 7, dtype: string


Resultado da limpeza:
- colunas: de 14 para 10 (as 4 `Unnamed` vazias foram removidas);
- linhas: 830.000 antes e depois, nenhum dado válido foi perdido;
- `#N/D`: de 3.650 para 0, agora identificados como `NÃO IDENTIFICADO`;
- duplicatas: 96.553 mantidas de forma justificada;
- nenhum valor ausente antes ou depois da limpeza.

# SPRINT 4: Análise

A base não tem valor monetário nem quantidade. Como cada linha é um item, criamos a medida
**itens por compra**, contando as linhas de cada `CO_ID`.

Atenção à unidade de cada resultado: **itens** (linhas da base), **compras** (`CO_ID`) e
**clientes** (`CL_ID`) são coisas diferentes.

In [12]:
# Uma linha por compra, com a quantidade de itens e o gênero do cliente
compras = (
    df_limpo.groupby("CO_ID")
    .agg(
        QTD_ITENS=("PR_ID", "size"),
        CL_GENERO=("CL_GENERO", "first"),
    )
    .reset_index()
)

resumo_itens = gerar_resumo(compras, "QTD_ITENS")

print("Estatísticas de itens por compra (unidade: compra)")
print("Compras com valor válido:", resumo_itens["contagem_valida"])
print("Média de itens por compra:", resumo_itens["media"])
print("Mínimo:", resumo_itens["minimo"])
print("Máximo:", resumo_itens["maximo"])

Estatísticas de itens por compra (unidade: compra)
Compras com valor válido: 18471
Média de itens por compra: 44.94
Mínimo: 1
Máximo: 89


### Interpretação (unidade: compra)
- As 18.471 compras têm, em média, **44,94 itens**: são compras grandes, típicas de abastecimento.
- O tamanho varia muito, de **1 item** (compra rápida) até **89 itens**.

## Filtro: compras acima da média de itens

In [13]:
compras_grandes = compras[compras["QTD_ITENS"] > resumo_itens["media"]]

print(f"Compras acima da média: {len(compras_grandes)} de {len(compras)} "
      f"({len(compras_grandes) / len(compras):.1%})")

Compras acima da média: 9306 de 18471 (50.4%)


9.306 das 18.471 compras (50,4%) têm mais itens que a média. A média fica bem no meio da
distribuição, sem um pequeno grupo de compras gigantes puxando o resultado.

## Agrupamento 1: categorias de produto

Para cada categoria contamos:
- **itens**: quantidade de linhas (unidades de produto registradas);
- **compras**: quantidade de compras distintas em que a categoria apareceu.

In [14]:
por_categoria = (
    df_limpo.groupby("PR_CAT")
    .agg(
        itens=("PR_ID", "size"),          # conta linhas
        compras=("CO_ID", "nunique"),     # conta compras distintas
    )
    .sort_values("itens", ascending=False)
)

por_categoria["pct_itens"] = (por_categoria["itens"] / len(df_limpo) * 100).round(1)
por_categoria["pct_compras"] = (por_categoria["compras"] / len(compras) * 100).round(1)

display(por_categoria)

,itens,compras,pct_itens,pct_compras
PR_CAT,,,,
ALIMENTOS,434767,18267,52.4,98.9
HIGIENE,155574,17610,18.7,95.3
LIMPEZA,145754,17475,17.6,94.6
BEBIDAS,43299,14735,5.2,79.8
PET,32399,13495,3.9,73.1
ACESSORIOS,14557,9224,1.8,49.9
NÃO IDENTIFICADO,3650,3228,0.4,17.5


ALIMENTOS concentra **52,4% dos itens** e aparece em **98,9% das compras**. HIGIENE e LIMPEZA
vêm em seguida (cerca de 18% dos itens cada) e também estão em mais de 94% das compras.
ACESSORIOS é a menor categoria: 1,8% dos itens, presente em cerca de metade das compras (49,9%).

## Agrupamento 2: compras por gênero

Usamos a tabela de compras (uma linha por compra) para contar quantas compras cada gênero
fez e qual o tamanho médio delas. Antes, conferimos se algum cliente aparece com mais de um gênero.

In [15]:
genero_inconsistente = (df_limpo.groupby("CL_ID")["CL_GENERO"].nunique() > 1).sum()
print("Clientes com gênero inconsistente:", genero_inconsistente)

por_genero = (
    compras.groupby("CL_GENERO")
    .agg(
        compras=("CO_ID", "size"),
        media_itens=("QTD_ITENS", "mean"),
    )
    .round(2)
)

display(por_genero)

Clientes com gênero inconsistente: 0


,compras,media_itens
CL_GENERO,,
F,9615,44.99
M,8856,44.88


Nenhum cliente tem gênero inconsistente. Clientes do gênero F fizeram **9.615 compras (52,1%)**
e os do gênero M, **8.856**. O tamanho médio é praticamente igual (44,99 e 44,88 itens):
o gênero não altera o tamanho da compra.

Nenhum agrupamento mede faturamento, pois a base não possui valores monetários.

# SPRINT 5: Exportação e Conclusões

Salvamos a base tratada sem o índice e lemos o arquivo de volta para conferir.

In [16]:
df_limpo.to_csv(ARQUIVO_SAIDA, index=False)

conferencia = pd.read_csv(ARQUIVO_SAIDA)
print("Registros antes:", len(df_original), "| depois:", len(conferencia))
print("Colunas exportadas:", conferencia.columns.tolist())

Registros antes: 830000 | depois: 830000
Colunas exportadas: ['DATA', 'CO_ID', 'CL_ID', 'CL_GENERO', 'CL_EC', 'CL_FHL', 'CL_SEG', 'PR_ID', 'PR_CAT', 'PR_NOME']


## Conclusões

1. A base tem **830.000 itens**, distribuídos em **18.471 compras** de **1.000 clientes**. Após a
   limpeza, todos os registros foram preservados; foram removidas só 4 colunas vazias.
2. Cada compra tem, em média, **44,94 itens** (mínimo 1, máximo 89), e **50,4%** das compras
   ficam acima dessa média.
3. **ALIMENTOS** representa **52,4% dos itens** e está em **98,9% das compras**; HIGIENE e LIMPEZA
   somam outros ~36% dos itens.
4. **ACESSORIOS** é a categoria menos frequente: 1,8% dos itens, presente em 49,9% das compras.
5. Clientes do gênero F fizeram **52,1% das compras**, mas o tamanho médio da compra é
   praticamente igual entre os gêneros (44,99 × 44,88 itens).

**Recomendação:** como ALIMENTOS está em quase todas as compras e ACESSORIOS em só metade,
vale testar ações de venda cruzada (exposição de acessórios perto de alimentos, combos)
para aumentar a presença dessa categoria. Também é recomendado corrigir na origem o
cadastro do produto 107, que aparece como `#N/D`.

**Limitação:** a base não tem valor monetário nem coluna de quantidade. Por isso não é possível
medir faturamento, e as linhas repetidas foram interpretadas como unidades a mais do mesmo
item, sem confirmação na fonte.